# [0] Mount Google Drive (RUN FIRST — checkpoint persistence)

**CRITICAL:** All training checkpoints are written directly to Google Drive so they survive Colab disconnects / runtime recycling. Run this cell before anything else and approve the authorization prompt.

In [ ]:
# Cell 0: Mount Google Drive so checkpoints persist across Colab disconnects
from google.colab import drive
drive.mount('/content/drive')

import os
# v2 = reason-then-decide grammar (<think> block). Versioned dir so auto-resume
# never accidentally resumes from an old-grammar checkpoint.
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/heartly_sft_model_v2"
DRIVE_FINAL_DIR = "/content/drive/MyDrive/heartly_final_v2"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_OUTPUT_DIR}")

# Show any existing checkpoints from a previous (interrupted) run
existing = sorted(os.listdir(DRIVE_OUTPUT_DIR)) if os.path.isdir(DRIVE_OUTPUT_DIR) else []
print(f"Existing contents: {existing if existing else '(empty — fresh run)'}")

# [1] Core Heartly Definitions and Utilities

Run this cell FIRST. It defines the tokenizer wrapper (`wrapper`), the knowledge-base classes (`KnowledgeUnit`, `KBOrganizer`), `NatureProfile`, `DatasetRenderer`, and the custom `collate_and_mask_loss` collator.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import hashlib
import json
from typing import Dict, List, Tuple, Set, Optional
import random

class HeartlyTokenizerWrapper:
    """
    Wraps a standard tokenizer to inject Heartly control tokens
    and handles formatting helper functions.
    """
    SPECIAL_TOKENS = {
        "additional_special_tokens": [
            "<think>", "</think>",
            "<decide>", "</decide>",
            "<verify>", "</verify>",
            "<stop>"
        ]
    }

    def __init__(self, base_model_id: str):
        print(f"Initializing base tokenizer: {base_model_id}")
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_id)

        # Add the Heartly specials
        num_added = self.tokenizer.add_special_tokens(self.SPECIAL_TOKENS)
        print(f"Added {num_added} special tokens to vocabulary.")

        # Explicitly set stop token as EOS equivalent for safety
        self.stop_token_id = self.tokenizer.convert_tokens_to_ids("<stop>")
        self.tokenizer.pad_token = self.tokenizer.eos_token

    def prepare_model_for_tokens(self, model: AutoModelForCausalLM):
        """Resizes the model embeddings to accommodate the new special tokens."""
        print("Resizing model token embeddings...")
        model.resize_token_embeddings(len(self.tokenizer))
        return model

    def format_response(self, decision: str, verification: str = None, answer: str = None, reasoning: str = None) -> str:
        """
        Formats outputs to match Heartly's exact grammar.

        NEW: reason-then-decide. The model first thinks inside <think>...</think>,
        THEN emits its decision — so 'speak or stay silent' is a reasoned choice,
        not a forced first token.
        """
        think_block = f"<think> {reasoning} </think>" if reasoning else ""

        if decision == "stop":
            return f"{think_block}<decide>stop</decide>"

        # "speak" path
        assert decision == "speak", "Decision must be 'speak' or 'stop'"
        assert verification in ["known", "unknown"], "Verification must be 'known' or 'unknown'"

        return f"{think_block}<decide>speak</decide><verify>{verification}</verify> {answer} <stop>"


# Initialize Heartly Tokenizer Wrapper (Moved here for correct execution order)
wrapper = HeartlyTokenizerWrapper("Qwen/Qwen2.5-0.5B")

class KnowledgeUnit:
    def __init__(self, entity: str, attribute: str, value: str, source: str):
        self.entity = entity.strip()
        self.attribute = attribute.strip()
        self.value = value.strip()
        self.source = source.strip()
        self.id = self._generate_hash()

    def _generate_hash(self) -> str:
        payload = f"{self.entity}|{self.attribute}|{self.value}|{self.source}"
        return hashlib.sha256(payload.encode('utf-8')).hexdigest()[:16]

    def to_dict(self):
        return {
            "id": self.id,
            "entity": self.entity,
            "attribute": self.attribute,
            "value": self.value,
            "source": self.source
        }

class KBOrganizer:
    """
    Maintains a closed, deduplicated, queryable knowledge base.
    Derives positive and boundary-negative pairs dynamically.
    """
    def __init__(self):
        self.store: Dict[str, KnowledgeUnit] = {}
        self.entities: Set[str] = set()
        self.attributes: Set[str] = set()
        # Fast indexes for complement queries
        self.entity_to_attrs: Dict[str, Set[str]] = {}

    def add_fact(self, entity: str, attribute: str, value: str, source: str) -> str:
        unit = KnowledgeUnit(entity, attribute, value, source)
        if unit.id not in self.store:
            self.store[unit.id] = unit
            self.entities.add(unit.entity)
            self.attributes.add(unit.attribute)

            if unit.entity not in self.entity_to_attrs:
                self.entity_to_attrs[unit.entity] = set()
            self.entity_to_attrs[unit.entity].add(unit.attribute)

        return unit.id

    def lookup(self, entity: str, attribute: str) -> Optional[str]:
        """Returns the value if present, else None."""
        for unit in self.store.values():
            if unit.entity.lower() == entity.lower() and unit.attribute.lower() == attribute.lower():
                return unit.value
        return None

class NatureProfile:
    def __init__(self, config: dict):
        self.name = config.get("name", "default-nature")
        self.abstain_ratio = config.get("abstain_ratio", 0.20)
        self.silence_ratio = config.get("silence_ratio", 0.05)
        self.say_less_ratio = config.get("say_less_ratio", 0.10)

class DatasetRenderer:
    def __init__(self, kb: KBOrganizer, profile: NatureProfile, tokenizer_wrapper: HeartlyTokenizerWrapper):
        self.kb = kb
        self.profile = profile
        self.tw = tokenizer_wrapper

    def generate_query_templates(self, entity: str, attribute: str) -> List[str]:
        """Generates realistic variations of queries for facts.

        If the attribute is already a full natural-language question (ends with '?'),
        use it directly instead of forcing it into the 'X of Y' template.
        """
        attr = attribute.strip()
        if attr.endswith("?"):
            base = attr
            return [
                base,
                f"Quick question: {base}",
                f"{base[:-1]}, please?"
            ]
        return [
            f"What is the {attribute} of {entity}?",
            f"Can you tell me the {attribute} of {entity}?",
            f"Provide the {attribute} for {entity}."
        ]

    # ---------- Reasoning templates (reason-then-decide) ----------
    def _reason_known(self, entity: str, attribute: str, source: str) -> str:
        return random.choice([
            f"The user asks about the {attribute} of {entity}. I have this fact in my knowledge (Source: {source}). I will speak.",
            f"This is a question about {entity}. I know this from {source}, so I can answer confidently. I will speak.",
            f"Checking my knowledge for the {attribute} of {entity}... found it (Source: {source}). I should answer.",
        ])

    def _reason_unknown(self, entity: str, attribute: str = None) -> str:
        what = f"the {attribute} of {entity}" if attribute else str(entity)
        return random.choice([
            f"The user asks about {what}. I have no information about this in my knowledge. I should say I don't know rather than guess.",
            f"Checking my knowledge for {what}... I find nothing. Guessing would risk being wrong, so I will admit I don't know.",
            f"This question concerns {what}, which is not in my knowledge base. The honest response is to say I don't have this information.",
        ])

    def _reason_silence(self, trigger: str) -> str:
        return random.choice([
            "The input is empty or contains no actual question. There is nothing meaningful to respond to. I will stay silent.",
            "This is just noise or a filler with no request behind it. No response is needed. I will stay silent.",
            "The user has not asked anything. Speaking now would add nothing. I choose to stay silent.",
        ])

    def render_dataset(self, extra_known_count: int = 0) -> List[Dict[str, str]]:
        """Renders the Heartly dataset from the KB.

        extra_known_count: number of direct 'known' samples (code/instructions/math)
        that will be merged in AFTER rendering — used so the abstain/silence ratios
        are balanced over the FULL final mix, not just the KB positives.
        """
        dataset = []

        # 1. Generate Positive Examples (Known)
        positive_samples = []
        for unit in self.kb.store.values():
            queries = self.generate_query_templates(unit.entity, unit.attribute)
            for q in queries:
                ans = f"The {unit.attribute} of {unit.entity} is {unit.value} (Source: {unit.source})."
                reasoning = self._reason_known(unit.entity, unit.attribute, unit.source)
                formatted_ans = self.tw.format_response(decision="speak", verification="known", answer=ans, reasoning=reasoning)
                positive_samples.append({"instruction": q, "output": formatted_ans})

        # 2. Generate Boundary Negatives (Unknown - same entities/attributes, but missing links)
        boundary_negatives = []

        # Attribute Mismatch: Select a known entity, query an attribute we don't have for it
        for entity in self.kb.entities:
            known_attrs = self.kb.entity_to_attrs[entity]
            missing_attrs = self.kb.attributes - known_attrs
            if missing_attrs:
                random_attr = random.choice(list(missing_attrs))
                queries = self.generate_query_templates(entity, random_attr)
                for q in queries:
                    ans = f"I do not have information about the {random_attr} of {entity}."
                    reasoning = self._reason_unknown(entity, random_attr)
                    formatted_ans = self.tw.format_response(decision="speak", verification="unknown", answer=ans, reasoning=reasoning)
                    boundary_negatives.append({"instruction": q, "output": formatted_ans})

        # Entity Mismatch: Query attributes on entities that don't exist in the KB at all
        # NOTE: only use short 'X of Y'-style attributes here (full-question attributes
        # don't make sense when re-targeted at another entity), and cap the sample size
        # so memory stays bounded with large multi-dataset KBs.
        unseen_entities = ["GPT-5", "Claude 4 Opus", "Gemini 2 Ultra", "Llama 4"]
        short_attrs = [a for a in self.kb.attributes if not a.strip().endswith("?")]
        max_attrs_per_entity = 10000
        for entity in unseen_entities:
            attrs_sample = random.sample(short_attrs, min(max_attrs_per_entity, len(short_attrs))) if short_attrs else []
            for attr in attrs_sample:
                queries = self.generate_query_templates(entity, attr)
                for q in queries:
                    ans = f"I do not have information about {entity}."
                    reasoning = self._reason_unknown(entity, attr)
                    formatted_ans = self.tw.format_response(decision="speak", verification="unknown", answer=ans, reasoning=reasoning)
                    boundary_negatives.append({"instruction": q, "output": formatted_ans})

        # Mix datasets according to target profile ratios.
        # FIX: ratios are computed over the FULL mix (KB positives + direct samples),
        # so heavy direct-sample merges no longer dilute abstention/silence behavior.
        num_pos = len(positive_samples) + extra_known_count
        target_neg = int(num_pos * (self.profile.abstain_ratio / (1 - self.profile.abstain_ratio))) if (1 - self.profile.abstain_ratio) > 0 else 0
        target_silence = int(num_pos * (self.profile.silence_ratio / (1 - self.profile.silence_ratio))) if (1 - self.profile.silence_ratio) > 0 else 0

        # 3. Generate Silence Examples
        # FIX: previously only 5 unique silence samples ever made it into the mix
        # (out of ~500k samples), so the model never learned <decide>stop</decide>.
        # Now we cycle a wide trigger list up to the FULL target count, with varied
        # reasoning so the behavior generalizes.
        silence_triggers = [
            "", " ", "...", "....", "..", "hey", "hi", "hello", "hello?", "yo",
            "hm", "hmm", "uh", "uhh", "um", "ok", "okay", "speak to me",
            "say something", "???", "!!", ".", ",", "nothing", "nevermind",
            "nvm", "just checking", "test", "are you there", "ping"
        ]
        silence_samples = []
        for i in range(target_silence):
            trigger = silence_triggers[i % len(silence_triggers)]
            reasoning = self._reason_silence(trigger)
            formatted_ans = self.tw.format_response(decision="stop", reasoning=reasoning)
            silence_samples.append({"instruction": trigger, "output": formatted_ans})

        random.shuffle(boundary_negatives)

        selected_neg = boundary_negatives[:target_neg]
        selected_silence = silence_samples  # already exactly target_silence long

        mix = positive_samples + selected_neg + selected_silence
        random.shuffle(mix)

        print(f"Compiled dataset. KB Positives: {len(positive_samples)}, Direct (external): {extra_known_count}, Boundary Negatives: {len(selected_neg)}, Silence: {len(selected_silence)}")
        return mix

def collate_and_mask_loss(batch, tokenizer, max_length=512):
    """
    Formats training inputs so that loss is ONLY calculated on the assistant output,
    but explicitly INCLUDES our decision and verification special tokens in the gradient.
    """
    input_ids_batch = []
    labels_batch = []
    skipped_items_count = 0

    for item_idx, item in enumerate(batch):
        instruction = None
        output = None

        # Try to get 'instruction' and 'output' directly
        if isinstance(item, dict) and 'instruction' in item and 'output' in item:
            instruction = item['instruction']
            output = item['output']
        elif isinstance(item, dict) and 'text' in item:
            full_text = item['text']
            # Expected format: "User: <instruction>\nAssistant: <output>"
            parts = full_text.split("Assistant: ", 1)
            if len(parts) == 2:
                # Remove "User: " prefix to get the instruction
                instruction = parts[0].replace("User: ", "").strip()
                output = parts[1]
            else:
                print(f"DEBUG: Skipping item {item_idx} because 'text' field does not match expected format: {full_text[:100]}...")
                skipped_items_count += 1
                continue
        else:
            print(f"DEBUG: Skipping item {item_idx} because of unexpected format: {item}")
            skipped_items_count += 1
            continue

        if not instruction or not output:
             print(f"DEBUG: Skipping item {item_idx} because instruction or output is empty after parsing: Instruction='{instruction}', Output='{output}'")
             skipped_items_count += 1
             continue

        prompt = f"User: {instruction}\nAssistant: "
        response = output

        # Tokenize prompt and full text
        prompt_tokens = tokenizer.encode(prompt, add_special_tokens=False)
        response_tokens = tokenizer.encode(response, add_special_tokens=False)

        full_tokens = prompt_tokens + response_tokens

        if not full_tokens:
            print(f"DEBUG: Skipping item {item_idx} because tokenization yielded empty full_tokens for: Instruction='{instruction}', Output='{output}'")
            skipped_items_count += 1
            continue

        # Truncate if necessary
        if len(full_tokens) > max_length:
            full_tokens = full_tokens[:max_length]

        # Labels: Mask prompt tokens with -100 (standard PyTorch cross-entropy ignore index)
        # Keep response tokens (including <decide>, </decide>, etc.) as active targets
        labels = [-100] * len(prompt_tokens) + response_tokens
        labels = labels[:max_length] # Ensure labels also truncated consistently

        # Pad up to max_length or dynamically per batch
        padding_len = max_length - len(full_tokens)
        if padding_len > 0:
            full_tokens += [tokenizer.pad_token_id] * padding_len
            labels += [-100] * padding_len

        input_ids_batch.append(full_tokens)
        labels_batch.append(labels)

    # Handle case where no valid items were processed
    if not input_ids_batch:
        print(f"DEBUG: Entire batch of size {len(batch)} was skipped by collate_and_mask_loss.")
        # Return empty tensors with correct dimensions if no items were processed
        return {
            "input_ids": torch.tensor([], dtype=torch.long).reshape(0, max_length) if max_length > 0 else torch.empty(0, dtype=torch.long),
            "labels": torch.tensor([], dtype=torch.long).reshape(0, max_length) if max_length > 0 else torch.empty(0, dtype=torch.long),
            "attention_mask": torch.tensor([], dtype=torch.long).reshape(0, max_length) if max_length > 0 else torch.empty(0, dtype=torch.long)
        }
    else:
        print(f"DEBUG: Collated a batch with {len(input_ids_batch)} valid items. Skipped {skipped_items_count} items.")


    return {
        "input_ids": torch.tensor(input_ids_batch, dtype=torch.long),
        "labels": torch.tensor(labels_batch, dtype=torch.long),
        "attention_mask": torch.tensor(input_ids_batch, dtype=torch.long).ne(tokenizer.pad_token_id).long()
    }

# [2] Initial Heartly Component Setup and Synthetic Dataset Generation

In [ ]:
# Cell 3: Re-initializing Knowledge Base and Rendering Dataset
# (Ensure Cell 2 with class definitions is run before this)

kb = KBOrganizer()
kb.add_fact("Llama 3", "release year", "2024", "Meta AI Announcement")
kb.add_fact("Llama 3", "developer", "Meta AI", "Meta AI Announcement")
kb.add_fact("Phi-3", "developer", "Microsoft", "MS Tech Blog")

config = {"name": "heartly-v2-test", "abstain_ratio": 0.30, "silence_ratio": 0.05}
profile = NatureProfile(config)

renderer = DatasetRenderer(kb, profile, wrapper)
final_dataset = renderer.render_dataset()

print(f"\nGenerated {len(final_dataset)} synthetic samples.")

# [3] Install `datasets` Library

Now that we've verified the basic functionality, let's proceed with downloading a dataset from Hugging Face and adapting it to your Knowledge Base (KB) format. I'll use a portion of the `squad` dataset as an example due to its question-answering structure.

First, we'll install the `datasets` library.

In [ ]:
# Cell 4
# Install the datasets library
%pip install datasets

# [4] Load Multiple Datasets (QA + Coding + Instructions + Math)

The model now trains on a **mix of 12 datasets** across four domains:

**Factual QA (routed through the Knowledge Base):**

| Dataset | Type | Samples used |
|---|---|---|
| SQuAD v1.1 | Wikipedia reading-comprehension QA | 50,000 |
| TriviaQA (rc.nocontext) | Open-domain trivia QA | 50,000 |
| Natural Questions Open | Real Google search questions | 30,000 |
| SciQ | Science exam QA | ~11,700 (full) |
| BoolQ | Yes/no questions over passages | ~9,400 (full) |
| WebQuestions | Freebase-grounded questions | ~3,800 (full) |

**Coding, instructions, and math (rendered as direct Heartly samples):**

| Dataset | Type | Samples used |
|---|---|---|
| CodeAlpaca-20k | Coding instructions (multi-language) | ~20,000 (full) |
| Python Code Instructions 18k | Python coding tasks | ~18,600 (full) |
| MBPP | Basic Python programming problems | ~370 (train) |
| Dolly-15k | Human-written general instructions | ~15,000 (full) |
| Alpaca | General instruction following | 30,000 |
| GSM8K | Grade-school math word problems | ~7,500 (full) |

In [ ]:
# Cell 5: Load MULTIPLE QA datasets (A100 EXTENDED MULTI-DATASET RUN)
from datasets import load_dataset

loaded_datasets = {}

def try_load(name, *args, **kwargs):
    """Load a dataset; if it fails (network / renamed repo), skip it gracefully."""
    try:
        ds = load_dataset(*args, **kwargs)
        loaded_datasets[name] = ds
        print(f"[OK]   {name}: {len(ds)} examples")
    except Exception as e:
        print(f"[SKIP] {name}: failed to load ({e})")

# 1. SQuAD v1.1 — Wikipedia reading comprehension
try_load("squad", 'rajpurkar/squad', split='train[:50000]')

# 2. TriviaQA (no-context config) — open-domain trivia
try_load("trivia_qa", 'mandarjoshi/trivia_qa', 'rc.nocontext', split='train[:50000]')

# 3. Natural Questions Open — real Google search queries
try_load("nq_open", 'google-research-datasets/nq_open', split='train[:30000]')

# 4. SciQ — science exam questions
try_load("sciq", 'allenai/sciq', split='train')

# 5. BoolQ — yes/no questions
try_load("boolq", 'google/boolq', split='train')

# 6. WebQuestions — Freebase-grounded questions
try_load("web_questions", 'stanfordnlp/web_questions', split='train')

# ---------- CODING DATASETS ----------
# 7. CodeAlpaca-20k — coding instructions across many languages
try_load("code_alpaca", 'sahil2801/CodeAlpaca-20k', split='train')

# 8. Python Code Instructions 18k — Python-focused coding tasks
try_load("python_code_18k", 'iamtarun/python_code_instructions_18k_alpaca', split='train')

# 9. MBPP — Mostly Basic Python Problems
try_load("mbpp", 'google-research-datasets/mbpp', 'full', split='train')

# ---------- GENERAL INSTRUCTION DATASETS ----------
# 10. Dolly-15k — human-written instruction/response pairs
try_load("dolly", 'databricks/databricks-dolly-15k', split='train')

# 11. Alpaca — general instruction following (subset)
try_load("alpaca", 'tatsu-lab/alpaca', split='train[:30000]')

# ---------- MATH DATASETS ----------
# 12. GSM8K — grade-school math word problems with reasoning
try_load("gsm8k", 'openai/gsm8k', 'main', split='train')

total = sum(len(d) for d in loaded_datasets.values())
print(f"\nTotal raw examples across {len(loaded_datasets)} datasets: {total}")


# [5] Populate KB with All Datasets and Render Heartly Dataset

Datasets are processed via two routes:

1. **Factual QA → Knowledge Base**: short-answer QA datasets are mapped to `(entity, attribute, value, source)` facts, and the renderer generates positives, boundary negatives (abstention), and silence samples from them.
2. **Coding / instructions / math → direct samples**: long-form answers (code, explanations, math reasoning) don't fit the `"The X of Y is Z"` fact template, so they are rendered **directly** as Heartly-formatted instruction/output pairs — now with a reasoning phase: `<think> ... I will speak. </think><decide>speak</decide><verify>known</verify> {answer} <stop>`.

The abstain/silence ratios are computed over the **full merged mix** (KB + direct samples), so the direct samples no longer dilute abstention and silence behavior.

In [ ]:
# Cell 6: Populate the KBOrganizer with facts from ALL loaded datasets
new_kb = KBOrganizer()

def extract_squad(ex):
    ans = ""
    if isinstance(ex.get('answers'), dict) and ex['answers'].get('text'):
        ans = ex['answers']['text'][0]
    entity = ex.get('title', '') or ex['context'].split('.')[0]
    return (entity, ex['question'], ans, "SQuAD")

def extract_trivia_qa(ex):
    ans = ex.get('answer', {}).get('value', '') if isinstance(ex.get('answer'), dict) else ""
    return ("general trivia", ex['question'], ans, "TriviaQA")

def extract_nq_open(ex):
    answers = ex.get('answer', [])
    ans = answers[0] if answers else ""
    q = ex['question'].strip()
    if not q.endswith('?'):
        q += '?'
    return ("general knowledge", q, ans, "Natural Questions")

def extract_sciq(ex):
    return ("science", ex['question'], ex.get('correct_answer', ''), "SciQ")

def extract_boolq(ex):
    q = ex['question'].strip()
    if not q.endswith('?'):
        q += '?'
    ans = "Yes" if ex.get('answer') else "No"
    return ("general knowledge", q, ans, "BoolQ")

def extract_web_questions(ex):
    answers = ex.get('answers', [])
    ans = answers[0] if answers else ""
    return ("general knowledge", ex['question'], ans, "WebQuestions")

KB_EXTRACTORS = {
    "squad": extract_squad,
    "trivia_qa": extract_trivia_qa,
    "nq_open": extract_nq_open,
    "sciq": extract_sciq,
    "boolq": extract_boolq,
    "web_questions": extract_web_questions,
}

# ---------- DIRECT-SAMPLE EXTRACTORS (coding / instructions / math) ----------
# These return (instruction, answer) pairs rendered straight into Heartly format,
# because long-form code / reasoning doesn't fit the 'X of Y is Z' fact template.

def extract_code_alpaca(ex):
    instr = ex.get('instruction', '').strip()
    inp = (ex.get('input') or '').strip()
    if inp:
        instr = f"{instr}\n\n{inp}"
    return (instr, ex.get('output', '').strip())

def extract_python_code_18k(ex):
    instr = ex.get('instruction', '').strip()
    inp = (ex.get('input') or '').strip()
    if inp:
        instr = f"{instr}\n\n{inp}"
    return (instr, ex.get('output', '').strip())

def extract_mbpp(ex):
    instr = ex.get('text', '').strip() or ex.get('prompt', '').strip()
    return (instr, ex.get('code', '').strip())

def extract_dolly(ex):
    instr = ex.get('instruction', '').strip()
    ctx = (ex.get('context') or '').strip()
    if ctx:
        instr = f"{instr}\n\nContext: {ctx}"
    return (instr, ex.get('response', '').strip())

def extract_alpaca(ex):
    instr = ex.get('instruction', '').strip()
    inp = (ex.get('input') or '').strip()
    if inp:
        instr = f"{instr}\n\n{inp}"
    return (instr, ex.get('output', '').strip())

def extract_gsm8k(ex):
    return (ex.get('question', '').strip(), ex.get('answer', '').strip())

DIRECT_EXTRACTORS = {
    "code_alpaca": extract_code_alpaca,
    "python_code_18k": extract_python_code_18k,
    "mbpp": extract_mbpp,
    "dolly": extract_dolly,
    "alpaca": extract_alpaca,
    "gsm8k": extract_gsm8k,
}

# Reasoning templates for direct samples (reason-then-decide)
DIRECT_REASONING = {
    "code_alpaca": "The user asks a coding task. I can work through this and produce the code. I will speak.",
    "python_code_18k": "This is a Python programming request. I know how to implement this. I will speak.",
    "mbpp": "This is a Python programming problem. I can solve it with a function. I will speak.",
    "dolly": "The user gives an instruction I can fulfill from my knowledge. I will speak.",
    "alpaca": "This is a task I can complete. I will work through it and respond. I will speak.",
    "gsm8k": "This is a math word problem. I can reason through it step by step. I will speak.",
}

direct_samples = []

for name, ds in loaded_datasets.items():
    if name in KB_EXTRACTORS:
        extractor = KB_EXTRACTORS[name]
        added = 0
        for example in ds:
            try:
                entity, attribute, value, source = extractor(example)
            except Exception:
                continue
            if entity and attribute and value:
                new_kb.add_fact(entity=entity, attribute=attribute, value=value, source=source)
                added += 1
        print(f"[KB]     {name}: added {added} facts")
    elif name in DIRECT_EXTRACTORS:
        extractor = DIRECT_EXTRACTORS[name]
        added = 0
        for example in ds:
            try:
                instruction, answer = extractor(example)
            except Exception:
                continue
            if instruction and answer:
                reasoning = DIRECT_REASONING.get(name, "I can answer this. I will speak.")
                formatted = wrapper.format_response(decision="speak", verification="known", answer=answer, reasoning=reasoning)
                direct_samples.append({"instruction": instruction, "output": formatted})
                added += 1
        print(f"[DIRECT] {name}: added {added} samples")
    else:
        print(f"No extractor for {name}, skipping.")

print(f"\nKBOrganizer populated with {len(new_kb.store)} facts | Direct samples: {len(direct_samples)}")

# Render the Heartly-compatible dataset from the KB, then merge in the direct samples.
# extra_known_count makes the abstain/silence ratios hold over the FULL final mix.
new_renderer = DatasetRenderer(new_kb, profile, wrapper)
heartly_squad_list = new_renderer.render_dataset(extra_known_count=len(direct_samples))
heartly_squad_list.extend(direct_samples)
random.shuffle(heartly_squad_list)
print(f"Combined dataset size (KB-rendered + direct): {len(heartly_squad_list)}")

from datasets import Dataset
heartly_squad_dataset = Dataset.from_list(heartly_squad_list)

# Add a 'text' column combining 'instruction' and 'output'
def add_text_field(example):
    prompt = f"User: {example['instruction']}\nAssistant: "
    response = example['output']
    example['text'] = prompt + response
    return example

heartly_squad_dataset = heartly_squad_dataset.map(add_text_field)

# --- Train / Eval split (95% / 5%) ---
split = heartly_squad_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split['train']
eval_dataset = split['test']
print(f"\nTrain samples: {len(train_dataset)} | Eval samples: {len(eval_dataset)}")

print("\n--- Sample rows ---")
for i in range(min(len(train_dataset), 3)):
    sample = train_dataset[i]
    print(f"\nSample #{i+1}:")
    print(f"Instruction: {sample['instruction']}")
    print(f"Output: {sample['output']}")


# [6] Demonstrate `collate_and_mask_loss` Functionality

### Demonstrating `collate_and_mask_loss`

Now that we have our `heartly_squad_dataset`, let's see how the `collate_and_mask_loss` function prepares a batch for training. This function ensures that only the assistant's response (including the special control tokens) contributes to the loss calculation during fine-tuning, while the user's prompt is masked.

In [ ]:
# Cell 7
from transformers import AutoTokenizer
import torch
from datasets import Dataset # Ensure Dataset is imported for clarity

# Re-initialize a tokenizer for demonstration of collate_and_mask_loss
# We'll use the same base model as the wrapper was initialized with
demo_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

# Add the special tokens to this demo tokenizer as well, as collate_and_mask_loss expects them
SPECIAL_TOKENS = {"additional_special_tokens": ["<think>", "</think>", "<decide>", "</decide>", "<verify>", "</verify>", "<stop>"]}
demo_tokenizer.add_special_tokens(SPECIAL_TOKENS)
demo_tokenizer.pad_token = demo_tokenizer.eos_token

# Take a small batch from the generated dataset
# Ensure `heartly_squad_dataset` is treated as a datasets.Dataset object
# and explicitly convert the selected subset to a list of dictionaries.
dummy_batch = heartly_squad_dataset.select(range(3)).to_list()

# Apply the collate_and_mask_loss function
processed_batch = collate_and_mask_loss(dummy_batch, demo_tokenizer)

print("\n--- Processed Batch (Inputs and Labels) ---")
for i in range(len(dummy_batch)):
    print(f"\nSample {i+1}:")
    print(f"Original Instruction: {dummy_batch[i]['instruction']}")
    print(f"Original Output: {dummy_batch[i]['output']}")

    # Decode input_ids for better readability (showing what the model sees)
    input_tokens_decoded = demo_tokenizer.decode(processed_batch['input_ids'][i], skip_special_tokens=False)
    print(f"Input Tokens (Decoded): {input_tokens_decoded}")

    # Show labels, highlighting masked (-100) vs. active tokens
    labels_print = [f"{t}({processed_batch['labels'][i][j]})" if processed_batch['labels'][i][j] != -100 else "-100"
                    for j, t in enumerate(processed_batch['input_ids'][i])]
    print(f"Labels (token_id/-100): {labels_print}")

# You can also inspect the tensor shapes:
print(f"\nShape of input_ids: {processed_batch['input_ids'].shape}")
print(f"Shape of labels: {processed_batch['labels'].shape}")
print(f"Shape of attention_mask: {processed_batch['attention_mask'].shape}")

# [7] SFTTrainer Setup and Training Initiation

### Setting up the EXTENDED Training Pipeline (A100-tuned)

10 epochs over a large multi-domain mix (~550k+ rendered samples: factual QA from SQuAD/TriviaQA/Natural Questions/SciQ/BoolQ/WebQuestions, coding from CodeAlpaca/Python-18k/MBPP, general instructions from Dolly/Alpaca, and math from GSM8K), effective batch size 32 (16 per device × 2 grad accum), bfloat16 mixed precision, cosine LR schedule with warmup, periodic eval on a 5% hold-out, checkpointing every 500 steps **directly to Google Drive**, and `load_best_model_at_end` so the best-eval-loss checkpoint is restored after training.

**Disconnect-proof:** checkpoints go to `/content/drive/MyDrive/heartly_sft_model`, and the training call auto-resumes from the latest checkpoint if one exists — if Colab disconnects, just reconnect, re-run all cells, and training continues where it left off.

**Heartly-aware evaluation:** in addition to `eval_loss`, a custom metric computes **`eval_control_accuracy`** — accuracy measured ONLY on the Heartly control tokens (`<think>`, `<decide>`, `speak`/`stop`, `<verify>`, `known`/`unknown`, `<stop>`). Best-checkpoint selection uses this metric, so the pinned "best" checkpoint is the one that makes the best speak/silence/known/unknown *decisions*, not just the lowest blended text loss.

Expected runtime on an **A100: roughly 4–8 hours** (5x the data and ~3x the epochs of the previous run). (On a T4 the same config would need a smaller batch size — reduce `per_device_train_batch_size` to 4 if you ever fall back to a T4.)

We use the plain `transformers.Trainer` with `remove_unused_columns=False` because our custom loss-masking collator needs the raw dataset columns.

In [ ]:
# Cell 8
# Install TRL and Accelerate
%pip install accelerate trl

In [ ]:
# Cell 9: EXTENDED fine-tuning run — A100-tuned configuration
#
# Uses plain `transformers.Trainer` with remove_unused_columns=False so our
# custom collator receives the raw 'instruction'/'output'/'text' columns.
# On an A100 the dtype auto-detection below picks torch.bfloat16.

from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
import torch

# 1. Load the base model
model_id = "Qwen/Qwen2.5-0.5B"
print(f"Loading base model: {model_id}")

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    model_dtype = torch.bfloat16
    print("Using torch.bfloat16 for model loading.")
elif torch.cuda.is_available():
    model_dtype = torch.float16
    print("Using torch.float16 for model loading (bfloat16 not supported).")
else:
    model_dtype = torch.float32
    print("Using torch.float32 for model loading (CUDA not available).")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=model_dtype)

# 2. Resize embeddings for the new Heartly special tokens
model = wrapper.prepare_model_for_tokens(model)

# 3. Heartly-aware evaluation metric
# We measure accuracy ONLY on control tokens (the decide/verify/think/stop grammar),
# and use that for best-checkpoint selection instead of blended eval_loss.
import numpy as np

CONTROL_TOKEN_STRINGS = [
    "<think>", "</think>", "<decide>", "</decide>",
    "<verify>", "</verify>", "<stop>",
]
control_token_ids = set(wrapper.tokenizer.convert_tokens_to_ids(CONTROL_TOKEN_STRINGS))
# Also include the decision/verification WORDS (speak/stop/known/unknown as they
# appear inside the tags). We collect every token id used by those words.
for word in ["speak", "stop", "known", "unknown"]:
    for ids in (wrapper.tokenizer.encode(word, add_special_tokens=False),):
        control_token_ids.update(ids)
control_token_ids.discard(None)
control_token_ids_arr = np.array(sorted(t for t in control_token_ids if isinstance(t, int) and t >= 0))
print(f"Tracking {len(control_token_ids_arr)} control token ids for eval_control_accuracy")

def preprocess_logits_for_metrics(logits, labels):
    """Reduce logits to predicted ids on-GPU so eval doesn't OOM on the vocab dim."""
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(eval_pred):
    preds, labels = eval_pred.predictions, eval_pred.label_ids
    # Causal LM shift: token at position i is predicted by logits at position i-1
    preds = preds[:, :-1]
    labels = labels[:, 1:]
    valid = labels != -100
    control_mask = np.isin(labels, control_token_ids_arr) & valid
    answer_mask = valid & ~control_mask

    metrics = {}
    if control_mask.sum() > 0:
        metrics["control_accuracy"] = float((preds[control_mask] == labels[control_mask]).mean())
    else:
        metrics["control_accuracy"] = 0.0
    if answer_mask.sum() > 0:
        metrics["answer_accuracy"] = float((preds[answer_mask] == labels[answer_mask]).mean())
    return metrics

# 4. Training arguments — A100 TUNED, EXTENDED TRAINING
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir=DRIVE_OUTPUT_DIR,        # persists on Google Drive — survives disconnects
    per_device_train_batch_size=16,     # A100 has 40GB VRAM — plenty of headroom
    gradient_accumulation_steps=2,      # effective batch size = 32
    num_train_epochs=10,                # EXTENDED: was 3 — much more training
    max_steps=-1,                       # run full epochs
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,                  # slightly shorter warmup for the longer run
    weight_decay=0.01,                  # regularization for the longer run
    bf16=use_bf16,                      # mixed-precision training on A100
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,                     # must match save_steps for load_best_model_at_end
    per_device_eval_batch_size=16,
    save_strategy="steps",
    save_steps=500,                     # frequent saves — a disconnect costs minutes, not hours
    save_total_limit=2,                 # ~3GB per checkpoint; keep Drive usage bounded
    load_best_model_at_end=True,        # restore best checkpoint at the end
    metric_for_best_model="eval_control_accuracy",  # Heartly-aware: best decide/verify behavior
    greater_is_better=True,
    dataloader_num_workers=2,
    report_to="none",
    push_to_hub=False,
    remove_unused_columns=False,        # CRITICAL for our custom collator
)

# 5. Initialize the Trainer
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=lambda data: collate_and_mask_loss(data, wrapper.tokenizer),
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    args=training_args,
)

# 6. Train (expect roughly 4-8 hours on an A100 for the extended run)
# AUTO-RESUME: if a checkpoint already exists on Drive (from an interrupted run),
# continue from it instead of starting over.
import glob
existing_ckpts = glob.glob(os.path.join(training_args.output_dir, "checkpoint-*"))
if existing_ckpts:
    print(f"\nFound existing checkpoints: {sorted(existing_ckpts)}")
    print("Resuming training from the latest checkpoint...")
    trainer.train(resume_from_checkpoint=True)
else:
    print("\nNo existing checkpoints found — starting EXTENDED SFT training from scratch")
    print("(A100-tuned, 10 epochs, multi-dataset mix)...")
    trainer.train()
print("SFT training complete.")


# [8] Post-Training Evaluation: Is Heartly Fully Functional?

We probe the trained model on three behavior categories:

1. **Known facts** (in the KB) → expect `<verify>known</verify>` with the correct answer
2. **Unknown facts** (NOT in the KB) → expect `<verify>unknown</verify>`
3. **Silence triggers** → expect `<decide>stop</decide>`

A per-category accuracy scorecard is printed at the end.

In [ ]:
# Cell 10: Behavioral evaluation scorecard
import random
import torch

model.eval()
device = model.device
tok = wrapper.tokenizer

def generate_response(instruction, max_new_tokens=160):
    prompt = f"User: {instruction}\nAssistant: "
    inputs = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
        )
    text = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    return text.strip()

# ---- Build probe sets ----
random.seed(123)
kb_units = list(new_kb.store.values())
known_probes = random.sample(kb_units, min(15, len(kb_units)))

unknown_probes = [
    ("GPT-5", "release year"),
    ("Claude 4 Opus", "developer"),
    ("Gemini 2 Ultra", "parameter count"),
    ("Llama 4", "context window"),
    ("Mistral Large 3", "release year"),
    ("The planet Zorbex", "population"),
    ("Captain Nemo's submarine", "top speed"),
    ("The 2087 Olympics", "host city"),
    ("Fluxion batteries", "inventor"),
    ("The Andromeda Council", "founding year"),
]

silence_probes = ["", "...", "hey", "hello?", "speak to me", "hm", "yo"]

# ---- Run probes ----
results = {"known": [0, 0], "unknown": [0, 0], "silence": [0, 0]}  # [correct, total]

print("=" * 70)
print("KNOWN-FACT PROBES (expect <verify>known</verify>)")
print("=" * 70)
for unit in known_probes:
    q = f"What is the {unit.attribute} of {unit.entity}?"
    resp = generate_response(q)
    ok = "<verify>known</verify>" in resp
    results["known"][0] += ok
    results["known"][1] += 1
    print(f"[{'PASS' if ok else 'FAIL'}] Q: {q[:70]}")
    print(f"       A: {resp[:120]}")

print("\n" + "=" * 70)
print("UNKNOWN-FACT PROBES (expect <verify>unknown</verify>)")
print("=" * 70)
for entity, attr in unknown_probes:
    q = f"What is the {attr} of {entity}?"
    resp = generate_response(q)
    ok = "<verify>unknown</verify>" in resp
    results["unknown"][0] += ok
    results["unknown"][1] += 1
    print(f"[{'PASS' if ok else 'FAIL'}] Q: {q[:70]}")
    print(f"       A: {resp[:120]}")

print("\n" + "=" * 70)
print("SILENCE PROBES (expect <decide>stop</decide>)")
print("=" * 70)
for trigger in silence_probes:
    resp = generate_response(trigger, max_new_tokens=80)  # room for the <think> block
    ok = "<decide>stop</decide>" in resp
    results["silence"][0] += ok
    results["silence"][1] += 1
    print(f"[{'PASS' if ok else 'FAIL'}] Trigger: '{trigger}' -> {resp[:80]}")

# ---- Scorecard ----
print("\n" + "#" * 70)
print("HEARTLY BEHAVIOR SCORECARD")
print("#" * 70)
for cat, (correct, total) in results.items():
    pct = 100 * correct / total if total else 0
    bar = "█" * int(pct // 5)
    print(f"{cat.upper():>10}: {correct}/{total}  ({pct:.0f}%)  {bar}")
overall_correct = sum(v[0] for v in results.values())
overall_total = sum(v[1] for v in results.values())
print(f"{'OVERALL':>10}: {overall_correct}/{overall_total}  ({100*overall_correct/overall_total:.0f}%)")


# [9] Interactive Chat

Type any question and see the model's raw Heartly-formatted response. Type `quit` to stop.

In [ ]:
# Cell 11: Interactive chat with the trained Heartly model
print("Heartly chat - type 'quit' to exit\n")
while True:
    user_in = input("You: ")
    if user_in.strip().lower() in ("quit", "exit"):
        print("Bye!")
        break
    resp = generate_response(user_in)
    print(f"Heartly: {resp}\n")


# [10] (Optional) Load a Saved Checkpoint Without Retraining

If the runtime died or you just want to evaluate/chat with an earlier model, load any Drive checkpoint here — no GPU-hours wasted on retraining. Works with old-grammar checkpoints too (e.g. `heartly_sft_model/checkpoint-19500`), they just won't emit `<think>` blocks.

In [ ]:
# Cell 10b: Load a checkpoint from Drive (skip if you just finished training above)
# Set this to the checkpoint you want to load:
CKPT_PATH = "/content/drive/MyDrive/heartly_sft_model_v2"  # or e.g. .../heartly_sft_model/checkpoint-19500

import os, glob
from transformers import AutoModelForCausalLM

# If CKPT_PATH is an output dir, pick the newest checkpoint-* inside it
if os.path.isdir(CKPT_PATH) and not os.path.exists(os.path.join(CKPT_PATH, "model.safetensors")):
    ckpts = sorted(glob.glob(os.path.join(CKPT_PATH, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[-1]))
    assert ckpts, f"No checkpoints found in {CKPT_PATH}"
    CKPT_PATH = ckpts[-1]

print(f"Loading model from: {CKPT_PATH}")
model = AutoModelForCausalLM.from_pretrained(CKPT_PATH, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
if torch.cuda.is_available():
    model = model.to("cuda")
model.eval()
print("Model loaded. Cells 10 (scorecard) and 11 (chat) can now be run against it.")


# [11] Save the Trained Model

Saves the final model + tokenizer directly to Google Drive so it survives the Colab session ending.

In [ ]:
# Cell 12: Save the final model and tokenizer DIRECTLY to Google Drive
save_dir = DRIVE_FINAL_DIR  # /content/drive/MyDrive/heartly_final (set in Cell 0)
trainer.save_model(save_dir)
wrapper.tokenizer.save_pretrained(save_dir)
print(f"Model and tokenizer saved to Google Drive: {save_dir}")

# Also keep a local copy on the Colab VM (optional, fast access)
local_dir = "./heartly_final"
trainer.save_model(local_dir)
wrapper.tokenizer.save_pretrained(local_dir)
print(f"Local copy saved to {local_dir}")
